# FASTT — main results on People's Speech

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huseyin-karaca/fastt/blob/main/notebooks/main_peoples_speech.ipynb)

This notebook reproduces the paper's main experiment on People's Speech. Every method's hyperparameters are selected on
a 25 % hold-out of the corpus; every method is then trained and tested on the same ten folds of a 5×2
cross-validation of the remaining clips, and the two FASTT variants are compared with every other method on those
folds.

The configuration is fixed in `fastt.configs.MAIN["peoples_speech"]` and the code is in `fastt.experiment`. The
features and the stored searches and folds are read from the Hugging Face dataset
[`huseyin-karaca/fastt`](https://huggingface.co/datasets/huseyin-karaca/fastt); nothing is written to it.

| Level | What runs | Needs |
|---|---|---|
| **1** | every table, from the searches and folds stored on the Hub | a CPU, minutes |
| **2** | every fold refitted from the published features (`RETUNE = True`: the search as well) | a CPU, hours |
| **3** | level 2 on features rebuilt from the audio: run `extract` first on the same machine, which writes them to a local folder that level 3 reads (`FASTT_HUB` names another place) | a GPU for `extract` |

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("fastt") is None:     # Colab: install the package (a local or Docker run already has it)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "fastt[paper] @ git+https://github.com/huseyin-karaca/fastt.git@main"], check=True)

In [2]:
from fastt.configs import MAIN
from fastt.experiment import MainExperiment

LEVEL = 1          # 1, 2 or 3 (see above)
RETUNE = False     # levels 2 and 3: run the hyperparameter search again

exp = MainExperiment(MAIN["peoples_speech"], level=LEVEL, retune=RETUNE)

People's Speech: level 1 — search read from the Hub, folds read from the Hub
experts  data2vec-audio-base, ESPnet Transformer, wav2vec2-base-960h
arms     FASTT-Alternating, FASTT-Boosted, Raw XGBoost (AdaSTT), Raw SDT, SelectKBest XGBoost, SelectKBest SDT, ROVER (weighted), ROVER (confidence-weighted), CN-MBR, Score-Fusion, Hard-Selector (plain CE), Hard-Selector (cost-sensitive CE), Soft-Selector (mixture), QE-Router


## 1. Hyperparameter spaces (Table 2)

The free hyperparameters of every method and their choices; every other parameter is fixed
(`fastt.configs.SPACES`). The spaces are the same on every corpus.

In [3]:
exp.spaces()

Table 2 — hyperparameter spaces
10 random draws per arm; parameters not listed are fixed
  [FASTT-Alternating  (18 free of 18)]
           parameter kind                                   choices / range
      transform_type  cat {nonlinear, linear, diagonal, low_rank, identity}
      num_iterations  cat                                     {6, 12, 3, 1}
        transform_lr  cat                             {0.001, 0.0001, 0.01}
     transform_steps  cat                                    {100, 25, 400}
        surrogate_lr  cat                                {0.01, 0.1, 0.001}
     surrogate_steps  cat                                    {50, 200, 400}
             lambda1  cat                               {0.1, 0.001, 1e-05}
             lambda2  cat                               {1e-05, 0.1, 0.001}
    sel_n_estimators  cat                                             {400}
   sel_learning_rate  cat                            {0.03, 0.3, 0.1, 0.01}
       sel_max_depth  cat                                     {5, 12, 8, 3}
       sel_subsample  cat                                   {1.0, 0.6, 0.8}
       sel_colsample  cat                              {0.6, 1.0, 0.3, 0.1}
sel_min_child_weight  cat                                             {1.0}
      sel_reg_lambda  cat                                             {3.0}
        weight_decay  cat                                            {0.01}
           out_ratio  cat                                   {1.0, 0.1, 0.3}
    bottleneck_ratio  cat                                            {0.05}
  [FASTT-Boosted  (15 free of 15)]
        parameter kind                                   choices / range
   transform_type  cat {identity, low_rank, linear, nonlinear, diagonal}
          lambda1  cat                                           {0.001}
          lambda2  cat                               {0.001, 1e-05, 0.1}
        out_ratio  cat                                             {0.1}
 bottleneck_ratio  cat                                 {0.15, 0.5, 0.05}
       num_rounds  cat                                      {6, 1, 4, 2}
        num_trees  cat                                    {24, 8, 16, 4}
            depth  cat                                               {7}
            lmbda  cat                        {0.01, 0.0001, 1.0, 1e-06}
           lmbda2  cat                                           {1e-07}
          adam_lr  cat                                          {0.0003}
adam_weight_decay  cat                        {0.001, 1e-07, 1e-05, 0.1}
       batch_size  cat                                  {512, 128, 2048}
         patience  cat                                      {40, 20, 10}
   t_weight_decay  cat                                           {1e-06}
  [Raw XGBoost (AdaSTT)  (7 free of 7)]
           parameter kind        choices / range
    sel_n_estimators  cat   {1000, 400, 150, 50}
   sel_learning_rate  cat {0.3, 0.1, 0.03, 0.01}
       sel_max_depth  cat                    {3}
       sel_subsample  cat        {1.0, 0.8, 0.6}
       sel_colsample  cat                  {1.0}
sel_min_child_weight  cat                 {16.0}
      sel_reg_lambda  cat       {3.0, 0.3, 30.0}
  [Raw SDT  (14 free of 15)]
        parameter kind              choices / range
   t_weight_decay  cat                      {1e-06}
          lambda1  cat                      {0.001}
          lambda2  cat                      {0.001}
        out_ratio  cat                        {0.1}
 bottleneck_ratio  cat                       {0.05}
       num_rounds  cat                 {1, 2, 4, 6}
        num_trees  cat                          {8}
            depth  cat                 {2, 7, 5, 3}
            lmbda  cat   {1e-06, 1.0, 0.0001, 0.01}
           lmbda2  cat                        {0.1}
          adam_lr  cat {0.001, 0.003, 0.0003, 0.01}
adam_weight_decay  cat                      {1e-07}
       batch_size  cat             {2048, 512, 128}
         patience  cat              

## 2. Data

The three experts' pooled encoder features, their transcripts, and the transcripts' word-error counters against
the reference, for every clip. A sample of the stored counters is recomputed from the stored transcripts as a
check.

In [4]:
exp.load()

14,299 utterances x 3 routable model(s), from cache .fastt-cache/store_6128bca59929edbd
texts for 3 model(s), from cache
100.0% of 1,200 (utterance, model) counters reproduce exactly


EnsembleStore(peoples_speech, 'mean', 14,299 utterances, 3 routable)

## 3. Hyperparameter search

A random 25 % of the clips is held out for the search. For every method, ten configurations are drawn at random
from its space (Table 2), each fitted on 70 % of the hold-out and scored on the remaining 30 %; the configuration
with the lowest corpus WER is kept. When several tie on it, the one the paper's run kept is used
(`fastt.configs.SELECTED`), after checking that it is among the tied best. The other 75 % of the clips are never
seen by the search.

In [5]:
exp.search()


=== best fastt_alternating   corpus WER 0.3927   gap closed +24.0%   (10 trials, 0.0 min)
{
  "num_iterations": 12,
  "out_ratio": 0.1,
  "sel_colsample": 1.0,
  "sel_learning_rate": 0.01,
  "sel_max_depth": 8,
  "sel_min_child_weight": 1.0,
  "sel_n_estimators": 400,
  "sel_reg_lambda": 3.0,
  "sel_subsample": 0.8,
  "surrogate_lr": 0.1,
  "surrogate_steps": 200,
  "transform_lr": 0.0001,
  "transform_steps": 100,
  "transform_type": "linear",
  "weight_decay": 0.01
}

=== best fastt_boosted   corpus WER 0.3928   gap closed +23.7%   (10 trials, 0.0 min)
{
  "adam_lr": 0.0003,
  "adam_weight_decay": 0.001,
  "batch_size": 512,
  "depth": 7,
  "epochs": 200,
  "lmbda": 1.0,
  "lmbda2": 1e-07,
  "num_rounds": 2,
  "num_trees": 8,
  "patience": 20,
  "transform_type": "identity"
}

=== best adastt   corpus WER 0.3999   gap closed +8.9%   (10 trials, 0.0 min)
{
  "sel_colsample": 1.0,
  "sel_learning_rate": 0.1,
  "sel_max_depth": 3,
  "sel_min_child_weight": 16.0,
  "sel_n_estimators": 4


=== best rover_w   corpus WER 0.3994   gap closed +10.0%   (10 trials, 0.0 min)
{
  "alpha": 1.0,
  "conf_source": "cn",
  "null_penalty": 0.5,
  "weight_by_conf": false,
  "weighting": "log_odds"
}

=== best rover_conf   corpus WER 0.4052   gap closed -2.2%   (10 trials, 0.0 min)
{
  "alpha": 0.25,
  "conf_source": "cn",
  "conf_temp": 0.25,
  "null_penalty": 0.25,
  "weight_by_conf": false,
  "weighting": "log_odds"
}

=== best cn_mbr   corpus WER 0.4152   gap closed -23.0%   (10 trials, 0.0 min)
{
  "candidates": "members",
  "conf_source": "probe",
  "decode": "hyp",
  "null_prior": 0.25,
  "prior": "inverse_wer",
  "temperature": 10.0,
  "use_conf": true
}

=== best score_fusion   corpus WER 0.4259   gap closed -45.3%   (10 trials, 0.0 min)
{
  "mode": "hard",
  "w_conf": 1.0,
  "w_consensus": -1.0,
  "w_length": -1.0
}

=== best argmin_clf   corpus WER 0.3997   gap closed +9.3%   (10 trials, 0.0 min)
{
  "alpha": 1.0
}

=== best probe   corpus WER 0.4042   gap closed +0.0%   (10


=== best qe_router   corpus WER 0.4039   gap closed +0.5%   (10 trials, 0.0 min)
{
  "alpha_decades": 4,
  "alpha_lo": 0.0001,
  "n_alphas": 16,
  "standardize": false,
  "target": "errors"
}
wrote .fastt-cache/runs/level1/hpt_peoples_speech_c17aaa8a.json  (8 arm(s))


partition,n_utts,share
tune,3575,0.250
eval,10724,0.750
fit,2502,0.175
val,1073,0.075


  on the tuning-score rows: bsm(data2vec) 0.4042   oracle 0.3562   gap 0.0480
  the 10,724 evaluation rows are never seen by a trial
  objectives   : corpus_wer (minimize) — crossval.METRICS['corpus_wer'], on the tuning-score rows
  14 arms x 10 trials, read from the Hub; 10,724 evaluation clips the search never saw


HPTRun('hpt_peoples_speech_379dd94f', 6 arm(s), best fastt_alternating 0.3927, eval on 10,724 rows)

In [6]:
exp.dataset()

statistic,peoples_speech
Total clips,14299
Features,2049
Hyperparameter hold-out,3575
Evaluation rows (5x2),10724
Train size (per fold),5362
Test size (per fold),5362
Hours (est.),28.1201
Reference words,286169


In [7]:
exp.selected()

label,trials,hold-out WER,selected configuration
FASTT-Alternating,10,0.3927,"{""num_iterations"": 12, ""out_ratio"": 0.1, ""sel_colsample"": 1.0, ""sel_learning_rate"": 0.01, ""sel_max_depth"": 8, ""sel_min_child_weight"": 1.0, ""sel_n_estimators"": 400, ""sel_reg_lambda"": 3.0, ""sel_subsample"": 0.8, ""surrogate_lr"": 0.1, ""surrogate_steps"": 200, ""transform_lr"": 0.0001, ""transform_steps"": 100, ""transform_type"": ""linear"", ""weight_decay"": 0.01}"
FASTT-Boosted,10,0.3928,"{""adam_lr"": 0.0003, ""adam_weight_decay"": 0.001, ""batch_size"": 512, ""depth"": 7, ""epochs"": 200, ""lmbda"": 1.0, ""lmbda2"": 1e-07, ""num_rounds"": 2, ""num_trees"": 8, ""patience"": 20, ""transform_type"": ""identity""}"
Raw XGBoost (AdaSTT),10,0.3999,"{""sel_colsample"": 1.0, ""sel_learning_rate"": 0.1, ""sel_max_depth"": 3, ""sel_min_child_weight"": 16.0, ""sel_n_estimators"": 400, ""sel_reg_lambda"": 30.0, ""sel_subsample"": 1.0}"
Raw SDT,10,0.4042,"{""adam_lr"": 0.001, ""adam_weight_decay"": 1e-07, ""batch_size"": 512, ""depth"": 5, ""epochs"": 200, ""lmbda"": 0.0001, ""lmbda2"": 0.1, ""num_rounds"": 2, ""num_trees"": 8, ""patience"": 20, ""transform_type"": ""identity""}"
SelectKBest XGBoost,10,0.4012,"{""k"": 32, ""sel_colsample"": 0.6, ""sel_learning_rate"": 0.01, ""sel_max_depth"": 3, ""sel_min_child_weight"": 16.0, ""sel_n_estimators"": 50, ""sel_reg_lambda"": 30.0, ""sel_subsample"": 1.0}"
SelectKBest SDT,10,0.4020,"{""adam_lr"": 0.001, ""adam_weight_decay"": 0.001, ""batch_size"": 2048, ""depth"": 3, ""epochs"": 200, ""k"": 32, ""lmbda"": 1.0, ""lmbda2"": 0.1, ""num_rounds"": 2, ""num_trees"": 16, ""patience"": 20, ""transform_type"": ""identity""}"
ROVER (weighted),10,0.3994,"{""alpha"": 1.0, ""conf_source"": ""cn"", ""null_penalty"": 0.5, ""weight_by_conf"": false, ""weighting"": ""log_odds""}"
ROVER (confidence-weighted),10,0.4052,"{""alpha"": 0.25, ""conf_source"": ""cn"", ""conf_temp"": 0.25, ""null_penalty"": 0.25, ""weight_by_conf"": false, ""weighting"": ""log_odds""}"
CN-MBR,10,0.4152,"{""candidates"": ""members"", ""conf_source"": ""probe"", ""decode"": ""hyp"", ""null_prior"": 0.25, ""prior"": ""inverse_wer"", ""temperature"": 10.0, ""use_conf"": true}"
Score-Fusion,10,0.4259,"{""mode"": ""hard"", ""w_conf"": 1.0, ""w_consensus"": -1.0, ""w_length"": -1.0}"


## 4. 5×2 cross-validation

Five repetitions, each splitting the evaluation clips into two halves with its own seed. Every method is trained
on one half and tested on the other, then the other way round: ten folds, the same for every method.

In [8]:
exp.cross_validate();

  runlog HIT  cv5x2 seed 11 — 15/15 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 22 — 15/15 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 33 — 15/15 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 44 — 15/15 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 55 — 15/15 group(s) cached, nothing refitted


5 repetitions x 2 folds x 24 arms, read from the Hub


## 5. Main results (Table 3)

Corpus WER and mean utterance WER (mean ± sd over the ten folds), the share of the oracle gap closed and the
selection accuracy at 0, 1 and 2 words of tolerance, and how often each method picks each expert.

In [9]:
exp.results()

Table 3 — main results, People's Speech
mean ± sd over 10 folds; GC / uGC / Acc in %; dist = share of clips routed to data2vec-audio-base / ESPnet Transformer / wav2vec2-base-960h; best non-oracle value per column in bold
  [individual]
                                    arm             WER            uWER        GC@0        GC@1        GC@2       uGC@0       uGC@1        uGC@2      Acc@0      Acc@1      Acc@2           dist
                    data2vec-audio-base 0.4011 ± 0.0029 0.5160 ± 0.0023   0.0 ± 0.0   0.0 ± 0.0   0.0 ± 0.0   0.0 ± 0.0   0.0 ± 0.0    0.0 ± 0.0 53.0 ± 0.3 75.3 ± 0.3 88.2 ± 0.3 1.00-0.00-0.00
                     ESPnet Transformer 0.4237 ± 0.0024 0.5336 ± 0.0028 -43.4 ± 3.1 -55.1 ± 4.2 -80.4 ± 6.9 -24.9 ± 1.5 -38.3 ± 2.6  -71.5 ± 5.4 43.5 ± 0.7 63.3 ± 0.3 76.9 ± 0.4 0.00-1.00-0.00
                     wav2vec2-base-960h 0.4084 ± 0.0021 0.5322 ± 0.0033 -14.0 ± 4.2 -17.8 ± 5.4 -26.0 ± 8.2 -22.9 ± 3.9 -35.2 ± 5.9 -65.9 ± 12.4 51.2 ± 0.4 71.3 ± 0.5 84.0 ± 0.5 0.00-0.00-1.00
Best-Single-Model (data2vec-audio-base) 0.4011 ± 0.0029 0.5160 ± 0.0023   0.0 ± 0.0   0.0 ± 0.0   0.0 ± 0.0   0.0 ± 0.0   0.0 ± 0.0    0.0 ± 0.0 53.0 ± 0.3 75.3 ± 0.3 88.2 ± 0.3 1.00-0.00-0.00
  [external]
                              arm             WER            uWER        GC@0        GC@1        GC@2       uGC@0       uGC@1       uGC@2      Acc@0      Acc@1      Acc@2           dist
                  Random-Ensemble 0.4102 ± 0.0022 0.5265 ± 0.0028 -17.3 ± 2.6 -22.0 ± 3.5 -32.2 ± 5.4 -14.9 ± 2.3 -22.9 ± 3.6 -42.7 ± 7.4 49.3 ± 0.4 70.2 ± 0.3 83.3 ± 0.3 0.33-0.33-0.33
                 ROVER (weighted) 0.3894 ± 0.0023 0.5209 ± 0.0033  22.4 ± 2.9  28.5 ± 3.5  41.4 ± 4.8  -6.9 ± 2.4 -10.6 ± 3.6 -19.7 ± 6.6          —          —          —              —
      ROVER (confidence-weighted) 0.3979 ± 0.0022 0.5298 ± 0.0034   6.3 ± 3.2   7.9 ± 4.1  11.5 ± 5.8 -19.6 ± 2.3 -30.1 ± 3.4 -56.1 ± 6.2          —          —          —              —
                           CN-MBR 0.4049 ± 0.0024 0.5328 ± 0.0041  -7.3 ± 3.4  -9.3 ± 4.4 -13.6 ± 6.6 -23.8 ± 3.1 -36.5 ± 4.5 -67.9 ± 7.6 51.5 ± 0.7 71.9 ± 0.2 84.6 ± 0.2 0.39-0.21-0.40
                     Score-Fusion 0.4247 ± 0.0030 0.5365 ± 0.0021 -45.3 ± 2.3 -57.5 ± 3.2 -83.8 ± 5.8 -29.1 ± 1.8 -44.7 ± 2.8 -83.5 ± 7.4 45.6 ± 0.3 64.7 ± 0.3 78.7 ± 0.5 0.34-0.36-0.29
         Hard-Selector (plain CE) 0.3930 ± 0.0024 0.5109 ± 0.0021  15.7 ± 2.5  19.9 ± 3.1  29.0 ± 4.3   7.3 ± 0.6  11.2 ± 1.0  20.8 ± 1.9 55.4 ± 0.4 77.0 ± 0.5 89.2 ± 0.4 0.92-0.04-0.05
Hard-Selector (cost-sensitive CE) 0.3992 ± 0.0045 0.5156 ± 0.0025   3.8 ± 4.1   4.9 ± 5.2   7.2 ± 7.7   0.6 ± 0.6   0.9 ± 0.9   1.6 ± 1.8 53.1 ± 0.4 75.4 ± 0.3 88.3 ± 0.3 1.00-0.00-0.00
                        QE-Router 0.3924 ± 0.0019 0.5117 ± 0.0031  16.9 ± 2.0  21.4 ± 2.4  31.1 ± 3.2   6.1 ± 1.8   9.4 ± 2.7  17.6 ± 5.1 57.5 ± 0.5 77.4 ± 0.4 88.9 ± 0.2 0.41-0.25-0.33
  [classical]
                 arm             WER            uWER       GC@0       GC@1       GC@2      uGC@0      uGC@1      uGC@2      Acc@0      Acc@1      Acc@2           dist
Raw XGBoost (AdaSTT) 0.3879 ± 0.0022 0.5025 ± 0.0034 25.5 ± 1.9 32.4 ± 2.3 47.1 ± 3.2 19.2 ± 2.1 29.5 ± 3.2 55.1 ± 6.6 60.6 ± 0.6 79.6 ± 0.4 90.3 ± 0.3 0.40-0.17-0.43
             Raw SDT 0.4011 ± 0.0029 0.5160 ± 0.0023  0.0 ± 0.0  0.0 ± 0.0  0.0 ± 0.0  0.0 ± 0.0  0.0 ± 0.0  0.0 ± 0.0 53.0 ± 0.3 75.3 ± 0.3 88.2 ± 0.3 1.00-0.00-0.00
 SelectKBest XGBoost 0.3933 ± 0.0024 0.5085 ± 0.0019 15.0 ± 2.7 19.1 ± 3.5 27.7 ± 4.8 10.7 ± 2.0 16.4 ± 3.0 30.5 ± 5.6 58.2 ± 0.9 77.8 ± 0.5 89.4 ± 0.3 0.49-0.14-0.37
     SelectKBest SDT 0.4011 ± 0.0029 0.5160 ± 0.0023  0.0 ± 0.0  0.0 ± 0.0  0.0 ± 0.0  0.0 ± 0.0  0.0 ± 0.0  0.0 ± 0.0 53.0 ± 0.3 75.3 ± 0.3 88.2 ± 0.3 1.00-0.00-0.00
  [fastt]
              arm             WER            uWER       GC@0       GC@1       GC@2      uGC@0      uGC@1      uGC@2      Acc@0      Acc@1      Acc@2           dist
    FASTT-Boosted 0.3860 ± 0.0024 0.5008 ± 0.0030 29.0 ± 1.8 36.8 ± 2.2 53.6 ± 2.7 21.6 

In [10]:
exp.routing()

model,n_picks,n_fused,share_data2vec,share_espnet_transformer,share_wav2vec2,top_member,share_top,entropy,entropy_max,share_sd_mean,switch_rate
achievable_oracle,53620,0,0.8824,0.0573,0.0603,data2vec,0.8824,0.6399,1.5850,0.0024,0.1176
adastt,53620,0,0.3996,0.1739,0.4264,wav2vec2,0.4264,1.4920,1.5850,0.0214,0.6004
argmin_clf,53620,0,0.9150,0.0376,0.0474,data2vec,0.9150,0.5037,1.5850,0.0088,0.0850
bsm,53620,0,1.0000,0.0000,0.0000,data2vec,1.0000,0.0000,1.5850,0.0000,0.0000
cn_mbr,53620,0,0.3898,0.2113,0.3988,wav2vec2,0.3988,1.5326,1.5850,0.0052,0.6102
data2vec,53620,0,1.0000,0.0000,0.0000,data2vec,1.0000,0.0000,1.5850,0.0000,0.0000
espnet_transformer,53620,0,0.0000,1.0000,0.0000,espnet_transformer,1.0000,0.0000,1.5850,0.0000,1.0000
fastt_alternating,53620,0,0.3485,0.2090,0.4424,wav2vec2,0.4424,1.5225,1.5850,0.0140,0.6515
fastt_boosted,53620,0,0.4373,0.1699,0.3928,data2vec,0.4373,1.4858,1.5850,0.0140,0.5627
kbest_sdt,53620,0,1.0000,0.0000,0.0000,data2vec,1.0000,0.0000,1.5850,0.0000,0.0000


## 6. Significance (Table 6)

Every method against the FASTT anchor (the better of the two FASTT variants on corpus WER) on the same ten folds:
Dietterich's 5×2 cv paired $t$ test and Alpaydın's combined 5×2 cv $F$ test, on corpus WER and on mean utterance
WER.

In [11]:
exp.tests()

Table 6 — every arm against the FASTT anchor, People's Speech
on the same 10 folds; positive effect = the anchor is better, in the metric's own units; p < 0.05 in bold. The F-test is the one the paper reports.
  [mean utterance WER — anchor FASTT-Boosted]
                       comparator  effect t_dietterich p_dietterich F_alpaydin p_alpaydin
              data2vec-audio-base +0.0152        9.080       0.0003     67.010     0.0001
               ESPnet Transformer +0.0328       18.984       0.0000    331.182     0.0000
               wav2vec2-base-960h +0.0313       10.279       0.0001    111.807     0.0000
                Best-Single-Model +0.0152        9.080       0.0003     67.010     0.0001
                  Random-Ensemble +0.0257       20.916       0.0000    388.085     0.0000
                 ROVER (weighted) +0.0201        7.729       0.0006     59.831     0.0001
      ROVER (confidence-weighted) +0.0290       11.143       0.0001    123.541     0.0000
                           CN-MBR +0.0319       10.073       0.0002     96.280     0.0000
                     Score-Fusion +0.0357       22.560       0.0000    468.718     0.0000
         Hard-Selector (plain CE) +0.0101        6.085       0.0017     29.056     0.0008
Hard-Selector (cost-sensitive CE) +0.0148        8.868       0.0003     60.627     0.0001
                        QE-Router +0.0109        8.816       0.0003     70.154     0.0001
             Raw XGBoost (AdaSTT) +0.0017        2.481       0.0558      3.455     0.0918
                          Raw SDT +0.0152        9.080       0.0003     67.010     0.0001
              SelectKBest XGBoost +0.0077        3.437       0.0185      9.179     0.0123
                  SelectKBest SDT +0.0152        9.080       0.0003     67.010     0.0001
                FASTT-Alternating +0.0013        2.370       0.0639      2.711     0.1413
            Oracle (no tolerance) -0.0551      -39.397       0.0000   1632.592     0.0000
        Oracle (1 word tolerance) -0.0306      -20.273       0.0000    459.663     0.0000
        Oracle (2 word tolerance) -0.0094       -4.729       0.0052     32.486     0.0006
  [corpus WER — anchor FASTT-Boosted]
                       comparator  effect t_dietterich p_dietterich F_alpaydin p_alpaydin
              data2vec-audio-base +0.0151       11.672       0.0001    110.882     0.0000
               ESPnet Transformer +0.0377       27.554       0.0000    750.607     0.0000
               wav2vec2-base-960h +0.0223       16.317       0.0000    268.228     0.0000
                Best-Single-Model +0.0151       11.672       0.0001    110.882     0.0000
                  Random-Ensemble +0.0241       47.435       0.0000   2214.111     0.0000
                 ROVER (weighted) +0.0034        2.179       0.0812      6.098     0.0297
      ROVER (confidence-weighted) +0.0118        7.782       0.0006     66.136     0.0001
                           CN-MBR +0.0189       12.855       0.0001    169.002     0.0000
                     Score-Fusion +0.0387       32.596       0.0000    962.801     0.0000
         Hard-Selector (plain CE) +0.0069        4.839       0.0047     29.999     0.0008
Hard-Selector (cost-sensitive CE) +0.0131        4.071       0.0096     10.673     0.0087
                        QE-Router +0.0063       10.227       0.0002     85.621     0.0001
             Raw XGBoost (AdaSTT) +0.0018        2.255       0.0738      3.686     0.0814
                          Raw SDT +0.0151       11.672       0.0001    110.882     0.0000
              SelectKBest XGBoost +0.0073        5.013       0.0041     20.721     0.0019
                  SelectKBest SDT +0.0151       11.672       0.0001    110.882     0.0000
                FASTT-Alternating +0.0019        2.866       0.0352      8.162     0.0159
            Oracle (no tolerance) -0.0369      -55.783       0.0000   3267.641     0.0000
        Oracle (1 word tolerance) -0.0259      -40.767       0.0000   1765.451     0.0000
        Oracle (

## 7. Record

The configuration, the selected hyperparameters and every table above, in `main_peoples_speech.json`;
`manuscript_tables` reads it from the folder `FASTT_RECORDS_DIR` names.

In [12]:
exp.save();

wrote main_peoples_speech.json
